# Agent Architecture
![Passenger Advocate Agent Evaluation](https://raw.githubusercontent.com/rkumar-bengaluru/passenger_advocate/refs/heads/main/assets/core_architecture1.png)
**_Below is the high level description of the agent architecture. Please note actual implementation will be minimal cut down version of above architecture to save time._**

## API Gateway
**Role**: The entry point for all incoming user requests (e.g., "What's the status of my flight?").

**Function**: Receives HTTP/API calls, routes them appropriately, handles rate limiting, load balancing, and request throttling.

**Why it matters**: Provides a unified interface and protects backend services from direct exposure.

## Cache
**Role**: Stores previously computed responses or frequently accessed data.

**Function**: Before processing a new query, the system checks the cache. If a matching result exists (e.g., a recent flight status lookup for the same flight), it returns the cached answer immediately — saving time and cost.

**Why it matters**: Reduces latency and avoids redundant LLM calls or database queries.

## LangGraph Orchestrator
**Role**: The central "brain" of the agent system — coordinates the entire workflow.

**Function**: Uses LangGraph (a framework for building stateful, multi-step agent workflows) to route the query through the appropriate sequence of steps: intent classification, tool selection, data retrieval, and response generation.

**Why it matters**: Replaces rigid hardcoded pipelines with a flexible, graph-based orchestration that can branch, loop, and conditionally execute steps.

## Flight Status Query (Tool/Function Module)
**Role**: A specialized tool/module that the orchestrator can invoke to fetch real-time flight data.

**Function**: Calls airline operational databases (**hive_metastore.defaul**) to retrieve flight status information (e.g., delays, gate changes, cancellations).

**Why it matters**: Provides the factual, up-to-date data grounding that the LLM needs — avoiding hallucinations.

## Data Retrieval (RAG / Knowledge Retrieval)
**Role**: Fetches relevant knowledge from internal documents, FAQs, policy manuals, or vector databases.

**Function**: Uses Retrieval-Augmented Generation (RAG) to find contextually relevant text chunks based on the user's query, then feeds them into the LLM for answer synthesis. For the implementation i will be using openai embedding model **text-embedding-ada-002** to generate the embeddings with dimension 1536 and **chunk size** of 1024 and **chunk overlap** of 256. For storage since i already have **qdrant cluster**.

**Future**: This RAG implementation can be enhanced with **re-ranking **using bi-directional encoder followed by **cross encoder** to retrive better quality context. Please note this does degrades the performance and hence need to understand the use case better of performance is a concern and potentially discover what is the **latency** because of this.

**Why it matters**: Gives the LLM access to proprietary or domain-specific knowledge beyond its training data.

## LLM Answer Synthesis
**Role**: The language model that generates the final natural-language response.

**Function**: Takes the retrieved data (from Flight Status Query, RAG, etc.) along with the original user query and synthesizes a coherent, helpful, and empathetic response (e.g., "Your flight was delayed by 45 minutes…"). 

**Future**: Since this an MVP, i **will not attempt re-try** or **fall back to secondary model**. Personally i have gemini or gpt sometimes throws 429 errors and hence this is definately needed for a production grade agents. **My be use SLM if required on failure**.

**Why it matters**: This is the conversational AI core — it transforms raw data into human-friendly communication.

## Policy Compliance Engine
**Role**: Ensures all responses and actions comply with airline policies, regulatory requirements, and business rules.

**Function**: Reviews the generated response (or proposed action) against a ruleset — e.g., "Don't promise compensation beyond policy limits," "Don't disclose other passengers' information."

**Implementation**: I may not have this implementation for the MVP, however the idea is to check against the **guidelines**, **policies** and **guardrails** to protect **IP/PII** information leakage. **PII and compliance is a deal breaker **in few of my earlier implemntation of the agentic development.

**Why it matters**: Prevents the agent from making commitments or statements that violate company policy or regulations.

## PII Protection (Personally Identifiable Information)

(Although it seems it is repeated in compliance, because of the criticality i think i would consider this a separate component altogether)

**Role**: Detects and masks/redacts sensitive personal data.

**Function**: Scans inputs and outputs for PII (names, passport numbers, credit card details, etc.) and either redacts, masks, or encrypts them before logging or displaying.

**Why it matters**: Critical for data privacy compliance (GDPR, CCPA, etc.) and protecting passenger data.

**NOTE**: In one of the HR agent implementation with **Service Now integration**, the agent has the capability to create leave request for the user, this may contain **PHI** which needs protection along with **GDPR compliance** hacks.

##  Logging & Auditing

**Role**: Records all interactions, decisions, and data accesses.

**Function**: Creates an immutable audit trail of every query, tool call, policy check, and response — useful for debugging, compliance audits, and improvement.

**Implementation**: I have not used **Langsmith**, however i have used **Langfuse** in the agentic development, one of the drawback was either it supports masking or unmasking, which means there is no way to get the original text. We can work around the hack with **RBAC control** mechanism of masking and unmasking using **Presidio**, but this is something to keep in mind.

**Why it matters**: Essential for accountability, regulatory compliance, and continuous system improvement.


Since the implementation will be a cut down version below is the simplied flow for this agent.
![Passenger Advocate Agent Simplied Flow](https://raw.githubusercontent.com/rkumar-bengaluru/passenger_advocate/refs/heads/main/assets/simplified_flow.png)

Not all components are discussed in detail however i think the above are the major components to talk about. Below we will review the evaluation strategy which can be employed to test this agent.


# Agent Evaluation Strategy
![Passenger Advocate Agent Evaluation](https://raw.githubusercontent.com/rkumar-bengaluru/passenger_advocate/refs/heads/main/assets/eval_architecture.png)


## Passenger Advocate Agent
**Role**: The system under test. This is the actual AI agent (built with the architecture described previously) that interacts with passengers. The evaluation strategy wraps around this agent to observe its behavior.

## Test Dataset
**Role**: The input data used to "quiz" the agent.

**Function**: A curated collection of simulated passenger queries, multi-turn conversations, and edge cases (e.g., angry passengers, complex flight changes). It ensures the agent is tested against a wide variety of realistic scenarios.

## Evaluation Engine (GEval)
**Role**: The central processing unit for the evaluation.

**Function**: GEval (Generative Evaluation) is a framework that uses LLMs to evaluate other LLMs. It takes the Test Dataset, runs it through the Agent, collects the outputs, and applies the Scoring Models to calculate the various metrics. It provides a structured, step-by-step reasoning process to determine why an answer is good or bad.

## Scoring Models
**Role**: The "Judges."

**Function**: These are specialized models (often powered by advanced LLMs like GPT-4) or deterministic algorithms that actually assign scores to the agent's outputs based on the metrics defined in the diagram. They act as automated human evaluators.

## Review Dashboard
**Role**: The user interface for human overseers.

**Function**: A visual platform where developers, QA engineers, and compliance teams can view the scores, read the agent's responses, see the reasoning of the Scoring Models, and identify where the agent is failing or succeeding.

**Possible Implementation**: DeepEvals (My architechture is inspired by this framework), Ragas, langSmith, Phoenix

##  Evaluation Metrics
These are the specific KPIs (Key Performance Indicators) used to measure the agent's performance, categorized into four distinct areas:

### Quality Metrics
Measures the intrinsic quality of the agent's reasoning and final output.

**Answer Relevancy**: Does the response directly address the passenger's question, or does it include unnecessary fluff?

**Argument Correctness**: Is the logical reasoning sound? If the agent says a flight is delayed, is its justification factually correct?

**Plan Quality**: When the agent decides on a course of action (e.g., "First check flight status, then check hotel availability"), is it a good, logical plan?

**Step Efficiency**: Is the agent taking unnecessary steps? (e.g., calling an API twice when once would suffice).

**Bias**: Does the agent show unfair preference or prejudice in its responses (e.g., treating VIP passengers more politely than economy passengers)?

### Contextual Metrics
Measures how well the agent uses the provided knowledge base (RAG) and handles the overall conversation.

**Contextual Precision**: When the agent retrieves documents/policies to form an answer, are the retrieved documents highly relevant, or is there noise?

**Contextual Recall**: Did the agent retrieve all the necessary documents needed to answer the question completely?
Contextual Relevancy: Overall, is the retrieved context relevant to the user's query?

**Conversation Completeness**: Over a multi-turn chat, did the agent address all the passenger's concerns, or did it drop the ball halfway through?

**Goal Accuracy**: Did the agent actually achieve the ultimate goal the passenger set out to accomplish (e.g., successfully changing the flight)?

### Turn-Metrics
Specifically drills down into multi-turn conversations, evaluating the agent's performance at each specific step.

**Turn Contextual Precision / Recall / Relevancy**: The same as the contextual metrics above, but measured on a per-turn basis. This helps identify if the agent loses context or makes retrieval errors in the middle of a long conversation.

### Safety Metrics
Measures the risk, compliance, and safety of the agent—critical for a regulated industry like airlines.

**Hallucination**: Is the agent making up false information (e.g., inventing a fake delay reason)?

**Knowledge Retention**: Does the agent remember what was said earlier in the conversation? (e.g., If the passenger gave their booking reference in Turn 1, does the agent still know it in Turn 4?)

**PII Leakage**: Does the agent accidentally expose or reveal sensitive passenger data (Passport numbers, credit cards) that shouldn't be shown?

**Tool Use & Correctness**: Did the agent call the right tools (APIs), and did it pass the correct parameters to them? (e.g., Trying to refund a ticket instead of changing it).

**Toxicity**: Does the agent generate offensive, rude, or harmful language?

## How It All Works Together

- The Test Dataset feeds simulated passenger queries into the Passenger Advocate Agent.

- The agent executes its Agent Workflow (thinking, retrieving data, using tools).
 
- The Evaluation Engine (GEval) captures the inputs, the workflow steps, and the outputs.
 
- The Scoring Models analyze this data and calculate scores for all the Quality, Contextual, Turn, and Safety Metrics.
 
- The results are sent to the Review Dashboard, allowing engineers to continuously monitor and improve the agent before and after it interacts with real passengers.

# Use case 
## My flight was delayed 3 hours. Do I get a meal voucher?

### Agent Build
```python


def build_agent():
    workflow = StateGraph(AgentState)

    workflow.add_node("intent_classification", intent_classification)
    workflow.add_node("get_flight_status", flight_status_query)
    workflow.add_node("get_policy_details", get_policy_details)
    workflow.add_node("summarize_answer", summarize_answer)
    workflow.add_node("policy_guardrails", policy_guardrails)
    workflow.add_node("pii_check", pii_check)

    # Start → Intent
    workflow.add_edge(START, "intent_classification")

    # Conditional routing from intent
    workflow.add_conditional_edges(
        "intent_classification",
        route_from_intent_from_analysis,
        {
            "get_flight_status": "get_flight_status",
            "get_policy_details": "get_policy_details",
            "summarize_answer": "summarize_answer"
        }
    )

    # Downstream edges
    workflow.add_edge("get_flight_status", "summarize_answer")
    workflow.add_edge("get_policy_details", "summarize_answer")
    workflow.add_edge("summarize_answer", "policy_guardrails")
    workflow.add_edge("policy_guardrails", "pii_check")
    workflow.add_edge("pii_check", END)

    return workflow.compile()

```

### RAG Context
    [1] (score=0.8439): ### Frontier Airlines’ Commitments for Controllable Delays
*   Rebook passenger on same airline at no additional cost for significant delays
*   Meal or meal cash/voucher when flight delay results in passenger waiting for 3 hours or more

Frontier Airlines does not commit to:
*   Rebook on partner airline or another airline with which it has an agreement at no additional cost for significant delays
*   Complimentary hotel accommodations for any passenger affected by an overnight delay
*   Complimentary ground transportation to and from hotel for any passenger affected by an overnight delay
*   Cash compensation when a delay results in passenger waiting for 3 hours or more from the scheduled departure time
*   Credit/travel voucher when delay results in passenger waiting for 3 hours or more from the scheduled departure time
*   Frequent flyer miles when delay results in passenger waiting for 3 hours or more from the scheduled departure time
[2] (score=0.8336): JetBlue Airways does not commit to:
*   Cash compensation when a delay results in passenger waiting for 3 hours or more from the scheduled departure time
*   Frequent flyer miles when delay results in passenger waiting for 3 hours or more from the scheduled departure time

### Obserbility and Monitoring Logs in Langfuse
![Passenger Advocate Agent Evaluation1](https://raw.githubusercontent.com/rkumar-bengaluru/passenger_advocate/refs/heads/main/assets//user_case_1_monitoring.png)

### Final Output
Agent: Whether you are entitled to a meal voucher depends on the airline and the cause of the delay.

For a **controllable delay** of 3 hours or more, the following airlines commit to providing a meal or a meal voucher:
*   Delta Air Lines [3]
*   Frontier Airlines [1]
*   JetBlue Airways [5]
*   Southwest Airlines [2]
*   United Airlines [4]


# Use case 2

### User: Since it was cancelled, is United required to provide me with a hotel?

### Agent:

![Passenger Advocate Agent Evaluation1](https://raw.githubusercontent.com/rkumar-bengaluru/passenger_advocate/refs/heads/main/assets//agent_response_2.png)

%md
# Use case 3

### User: What was the status of United flight UA123 on 2023-01-01??

### Agent:

![Passenger Advocate Agent Evaluation1](https://raw.githubusercontent.com/rkumar-bengaluru/passenger_advocate/refs/heads/main/assets//agent_response_3.png)

### Obsersability & Monitoring
![Passenger Advocate Agent Evaluation1](https://raw.githubusercontent.com/rkumar-bengaluru/passenger_advocate/refs/heads/main/assets//monitoring_3.png)
### Hacks
The tool call is hacked here to return fixed json response to derive the correct output

In [0]:
%pip install databricks-langchain langchain-core langgraph
%restart_python

  Using cached protobuf-5.29.6-cp38-abi3-manylinux2014_x86_64.whl.metadata (592 bytes)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached protobuf-5.29.6-cp38-abi3-manylinux2014_x86_64.whl (320 kB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/131.6 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.7.0
    Uninstalling urllib3-2.7.0:
      Successfully uninstalled urllib3-2.7.0
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting un

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os
# Set environment variable
# gemini
os.environ["GOOGLE_API_KEY"] = "xxxx"

# qdrant
os.environ["TEMPERATURE"]='0.7'
os.environ["QDRANT_API_KEY"] = "xxx"
os.environ["QDRANT_CONNECTION_STRING"] = "https://7e8a5eae-2206-4ca6-96f5-678bf3369919.us-east4-0.gcp.cloud.qdrant.io:6334"
os.environ["GOOGLE_API_KEY"] = "xxx"

# open ai
os.environ["OPENAI_API_KEY"] = "xxx"
os.environ["OPENAI_EMBEDDING_MODEL"] = "text-embedding-ada-002"
os.environ["OPENAI_EMBEDDING_DIMENSION"] = "1536"
os.environ["OPENAI_EMBEDDING_DISTANCE"] = "Cosine"
os.environ["OPENAI_CHUNK_SIZE"] = "1024"
os.environ["OPENAI_CHUNK_OEVRLAP"] = "256"

# langfuse
os.environ["LANGFUSE_SECRET_KEY"] = "xxx"
os.environ["LANGFUSE_PUBLIC_KEY"] = "xxx"
os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"

DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sDATABRICKS_SHELL_DO_EXECUTE_START
n
n
n
n
e

DEBUG:py4j.clientserver:Answer received: !yv
DEBUG:py4j.clientserver:Command to send: c
o0
contains
sspark.databricks.workspaceUrl
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o0
get
sspark.databricks.workspaceUrl
e

DEBUG:py4j.clientserver:Answer received: !ysdbc-68428b72-3365.cloud.databricks.com
DEBUG:py4j.clientserver:Command to send: c
o0
contains
sspark.databricks.clusterUsageTags.sparkVersion
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o0
get
sspark.databricks.clusterUsageTags.sparkVersion
e

DEBUG:py4j.clientserver:Answer received: !ys16.4.x-photon-scala2.13
INFO:py4j.clientserver:Received command c on object id p0
DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sDATABRICKS_SHELL_DO_EXECUTE_END
n
n
n
n
e

DEBUG:py4j.clientserver:Answer recei

In [0]:
%pip install "opentelemetry-api>=1.33.1,<2" "opentelemetry-sdk>=1.33.1,<2"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install langfuse==4.6.1 google-genai==2.0 langchain_openai=1.2.1 qdrant_client=1.17.1 langchain_qdrant=1.1.0

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# use case 2 - Since it was cancelled, is United required to provide me with a hotel?
from langchain_core.messages import HumanMessage, AIMessage
from src.agent import agent_app
from src.utils import get_input_with_timeout,run_agent_loop
import traceback
from dotenv import load_dotenv
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from src.retrieval import OpenAIEmbeddingModel

load_dotenv()
# langfuse client for observability & monitoring.
langfuse_handler = CallbackHandler()

for user_input in run_agent_loop(agent_app):
    try:
        initial_state = {"messages": [HumanMessage(content=user_input)]}

        config = {
            "callbacks": [langfuse_handler],
            "run_name": "PassengerAdvocateAgent",           
            "metadata": {                                   
                "user_query": user_input,
                "environment": "development"
            }
        }

        for event in agent_app.stream(initial_state, config=config):
            # Print agent response
            for value in event.values():
                if "messages" in value and value["messages"]:
                    last_msg = value["messages"][-1]
                    if isinstance(last_msg, AIMessage):
                        print(f"Agent: {last_msg.content}")
                    elif isinstance(last_msg, HumanMessage):
                        print(f"User: {last_msg.content}")
        get_client().flush()

    except Exception as e:
        print(f"Error: {e}")
        traceback.print_exc()

DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sDATABRICKS_SHELL_DO_EXECUTE_START
n
n
n
n
e

DEBUG:py4j.clientserver:Answer received: !yv
DEBUG:py4j.clientserver:Command to send: c
o0
contains
sspark.databricks.workspaceUrl
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o0
get
sspark.databricks.workspaceUrl
e

DEBUG:py4j.clientserver:Answer received: !ysdbc-68428b72-3365.cloud.databricks.com
DEBUG:py4j.clientserver:Command to send: c
o0
contains
sspark.databricks.clusterUsageTags.sparkVersion
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o0
get
sspark.databricks.clusterUsageTags.sparkVersion
e

DEBUG:py4j.clientserver:Answer received: !ys16.4.x-photon-scala2.13
INFO:py4j.clientserver:Received command c on object id p0
DEBUG:langfuse:Thread: Media upload consumer thread #0 started and actively processing queue items
DEBUG:langfuse:Prompt cache initialized.
DEBUG:langf

--- Rearc AI Quest: Support Agent ---
Type 'quit' to exit.



User:  Since it was cancelled, is United required to provide me with a hotel?

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
DEBUG:langfuse:Event: on_chain_start, run_id: 019e39f2-7082-77d0-b1f7-cf736aed8dc9, parent_run_id: None
DEBUG:langfuse:Propagated 2 attributes to span '67640bf63ddfb226': {'langfuse.trace.metadata.user_query': 'Since it was cancelled, is United required to provide me with a hotel?', 'langfuse.trace.metadata.environment': 'development'}
DEBUG:langfuse:Event: on_chain_start, run_id: 019e39f2-7088-7e01-bdec-003fbc891ec9, parent_run_id: 019e39f2-7082-77d0-b1f7-cf736aed8dc9
DEBUG:langfuse:Propagated 2 attributes to span '07038b610caaa5db': {'langfuse.trace.metadata.user_query': 'Since it was cancelled, is United required to provide me with a hotel?', 'langfuse.trace.metadata.environment': 'development'}
DEBUG:langfuse:Propagated 2 attributes to span '05c6ca0ca55b

User: Since it was cancelled, is United required to provide me with a hotel?


...


User:  quit

DEBUG:py4j.clientserver:Command to send: m
d
o589
e

DEBUG:py4j.clientserver:Answer received: !yv
DEBUG:py4j.clientserver:Command to send: m
d
o590
e

DEBUG:py4j.clientserver:Answer received: !yv
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0



👋 Session ended.


[Trace(trace_id=tr-82d1c3c8a3da0838654b7a5f6415b2e5), Trace(trace_id=tr-c1ce54237fc7bb85f39a181308eaf83b), Trace(trace_id=tr-9e35dfb74b224e92ec653afc396dcae9), Trace(trace_id=tr-3e948ad57addce1ef0bb5cc3b2827386)]

DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sDATABRICKS_SHELL_DO_EXECUTE_END
n
n
n
n
e

DEBUG:py4j.clientserver:Answer received: !yv


## Evidence from langfuse instance

## Trace
![Passenger Advocate Agent Evaluation1](https://raw.githubusercontent.com/rkumar-bengaluru/passenger_advocate/refs/heads/main/assets//usecase_2_evidence.png)

In [0]:
# use case 3

from langchain_core.messages import HumanMessage, AIMessage
from src.agent import agent_app
from src.utils import get_input_with_timeout,run_agent_loop
import traceback
from dotenv import load_dotenv
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from src.retrieval import OpenAIEmbeddingModel

load_dotenv()
# langfuse client for observability & monitoring.
langfuse_handler = CallbackHandler()

for user_input in run_agent_loop(agent_app):
    try:
        initial_state = {"messages": [HumanMessage(content=user_input)]}

        config = {
            "callbacks": [langfuse_handler],
            "run_name": "PassengerAdvocateAgent",           
            "metadata": {                                   
                "user_query": user_input,
                "environment": "development"
            }
        }

        for event in agent_app.stream(initial_state, config=config):
            # Print agent response
            for value in event.values():
                if "messages" in value and value["messages"]:
                    last_msg = value["messages"][-1]
                    if isinstance(last_msg, AIMessage):
                        print(f"Agent: {last_msg.content}")
                    elif isinstance(last_msg, HumanMessage):
                        print(f"User: {last_msg.content}")
        get_client().flush()

    except Exception as e:
        print(f"Error: {e}")
        traceback.print_exc()

DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sDATABRICKS_SHELL_DO_EXECUTE_START
n
n
n
n
e

DEBUG:py4j.clientserver:Answer received: !yv
DEBUG:py4j.clientserver:Command to send: c
o0
contains
sspark.databricks.workspaceUrl
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o0
get
sspark.databricks.workspaceUrl
e

DEBUG:py4j.clientserver:Answer received: !ysdbc-68428b72-3365.cloud.databricks.com
DEBUG:py4j.clientserver:Command to send: c
o0
contains
sspark.databricks.clusterUsageTags.sparkVersion
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o0
get
sspark.databricks.clusterUsageTags.sparkVersion
e

DEBUG:py4j.clientserver:Answer received: !ys16.4.x-photon-scala2.13
INFO:py4j.clientserver:Received command c on object id p0


--- Rearc AI Quest: Support Agent ---
Type 'quit' to exit.



User:  My flight was delayed 3 hours. Do I get a meal voucher?

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
DEBUG:langfuse:Event: on_chain_start, run_id: 019e39f7-b7ca-7ae2-9048-e1e039f6ef0e, parent_run_id: None
DEBUG:langfuse:Propagated 

User: My flight was delayed 3 hours. Do I get a meal voucher?


...


User:  quit

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0



👋 Session ended.


[Trace(trace_id=tr-7fd58a918fc58936a77bdb8bcea878b8), Trace(trace_id=tr-17ea1b2372852db38432c02a26d89994), Trace(trace_id=tr-efcc26b1d36612c0f284f7a53346d051), Trace(trace_id=tr-60ee191af3d0567311ffb319acf1fd09)]

DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sDATABRICKS_SHELL_DO_EXECUTE_END
n
n
n
n
e

INFO:py4j.clientserver:Received command c on object id p0
DEBUG:py4j.clientserver:Answer received: !yv


## Use case 3 

## Evidence from langfuse
![Passenger Advocate Agent Evaluation](https://raw.githubusercontent.com/rkumar-bengaluru/passenger_advocate/refs/heads/main/assets/usecase_3_evidence.png)


In [0]:
# test data source

from src.flight_status import get_flight_status

response = get_flight_status("OO", "4277", "2023-12-30")

print(response)

DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sDATABRICKS_SHELL_DO_EXECUTE_START
n
n
n
n
e

DEBUG:py4j.clientserver:Answer received: !yv
DEBUG:py4j.clientserver:Command to send: c
o0
contains
sspark.databricks.workspaceUrl
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o0
get
sspark.databricks.workspaceUrl
e

DEBUG:py4j.clientserver:Answer received: !ysdbc-68428b72-3365.cloud.databricks.com
DEBUG:py4j.clientserver:Command to send: c
o0
contains
sspark.databricks.clusterUsageTags.sparkVersion
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o0
get
sspark.databricks.clusterUsageTags.sparkVersion
e

DEBUG:py4j.clientserver:Answer received: !ys16.4.x-photon-scala2.13
INFO:py4j.clientserver:Received command c on object id p0
DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sAUTORELOAD_SUCCESS
n
n
n
n
e

DEBUG:py4j.clientserver:Answer received: !yv



    --- Flight Query Parameters ---
    Airline Code     : OO
    Flight Number    : 4277
    Flight Date (raw): 2023-12-30
    Flight Date Hive : 2023-12-30
    --------------------------------
    
{'tool': 'get_flight_status', 'input': {'airline': 'OO', 'flight_number': '4277', 'flight_date': '2023-12-30'}, 'status': 'success', 'summary': 'Flight Status Summary:\n        Flight OO 4277 from Salt Lake City to Cedar City on 2023-12-30 was CANCELLED\n        '}


DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sDATABRICKS_SHELL_DO_EXECUTE_END
n
n
n
n
e

DEBUG:py4j.clientserver:Answer received: !yv


In [0]:
# use case 1

from langchain_core.messages import HumanMessage, AIMessage
from src.agent import agent_app
from src.utils import get_input_with_timeout,run_agent_loop
import traceback
from dotenv import load_dotenv
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from src.retrieval import OpenAIEmbeddingModel

load_dotenv()
# langfuse client for observability & monitoring.
langfuse_handler = CallbackHandler()

for user_input in run_agent_loop(agent_app):
    try:
        initial_state = {"messages": [HumanMessage(content=user_input)]}

        config = {
            "callbacks": [langfuse_handler],
            "run_name": "PassengerAdvocateAgent",           
            "metadata": {                                   
                "user_query": user_input,
                "environment": "development"
            }
        }

        for event in agent_app.stream(initial_state, config=config):
            # Print agent response
            for value in event.values():
                if "messages" in value and value["messages"]:
                    last_msg = value["messages"][-1]
                    if isinstance(last_msg, AIMessage):
                        print(f"Agent: {last_msg.content}")
                    elif isinstance(last_msg, HumanMessage):
                        print(f"User: {last_msg.content}")
        get_client().flush()

    except Exception as e:
        print(f"Error: {e}")
        traceback.print_exc()

DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sDATABRICKS_SHELL_DO_EXECUTE_START
n
n
n
n
e

DEBUG:py4j.clientserver:Answer received: !yv
DEBUG:py4j.clientserver:Command to send: c
o0
contains
sspark.databricks.workspaceUrl
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o0
get
sspark.databricks.workspaceUrl
e

DEBUG:py4j.clientserver:Answer received: !ysdbc-68428b72-3365.cloud.databricks.com
DEBUG:py4j.clientserver:Command to send: c
o0
contains
sspark.databricks.clusterUsageTags.sparkVersion
e

DEBUG:py4j.clientserver:Answer received: !ybtrue
DEBUG:py4j.clientserver:Command to send: c
o0
get
sspark.databricks.clusterUsageTags.sparkVersion
e

DEBUG:py4j.clientserver:Answer received: !ys16.4.x-photon-scala2.13
INFO:py4j.clientserver:Received command c on object id p0
DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sAUTORELOAD_SUCCESS
n
n
n
n
e

DEBUG:py4j.clientserver:Answer received: !yv


--- Rearc AI Quest: Support Agent ---
Type 'quit' to exit.



User:  What was the status of OO flight 4277 on 2023-12-30?

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clie

User: What was the status of OO flight 4277 on 2023-12-30?

    --- Flight Query Parameters ---
    Airline Code     : OO
    Flight Number    : 4277
    Flight Date (raw): 2023-12-30
    Flight Date Hive : 2023-12-30
    --------------------------------
    
User: What was the status of OO flight 4277 on 2023-12-30?

    You are a Passenger Advocate Agent. The user asked: "What was the status of OO flight 4277 on 2023-12-30?"

    ## Tool Output Context
    {
  "tool": "get_flight_status",
  "input": {
    "airline": "OO",
    "flight_number": "4277",
    "flight_date": "2023-12-30"
  },
  "status": "success",
  "summary": "\n        Flight Status Summary:\n        Flight OO 4277 from Salt Lake City to Cedar City on 2023-12-30 was CANCELLED\n        "
}

    ## Task
    Summarize the above tool output check the summary and dervice a clear, user-facing answer.
    - Mention the flight status, delays, cancellations, or policies as appropriate.
    - Use plain language, not JSON.
    - K

...


User:  quit

DEBUG:py4j.clientserver:Command to send: m
d
o787
e

DEBUG:py4j.clientserver:Answer received: !yv
DEBUG:py4j.clientserver:Command to send: m
d
o788
e

DEBUG:py4j.clientserver:Answer received: !yv
DEBUG:py4j.clientserver:Command to send: m
d
o789
e

DEBUG:py4j.clientserver:Answer received: !yv
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.


👋 Session ended.


[Trace(trace_id=tr-95c78d474eeef79989e6a010d233e060), Trace(trace_id=tr-9d30c7d118c4d71e9d2acf92d5aabf0a)]

DEBUG:py4j.clientserver:Command to send: c
o467
logExecuteCommandEvent
sDATABRICKS_SHELL_DO_EXECUTE_END
n
n
n
n
e

DEBUG:py4j.clientserver:Answer received: !yv


## Use case 1
Query - What was the status of OO flight 4277 on 2023-12-30?

## Use case 1 evidence in langfuse

![Passenger Advocate Agent Evaluation1](https://raw.githubusercontent.com/rkumar-bengaluru/passenger_advocate/refs/heads/main/assets//usecase_1_evidence.png)

In [0]:
%sql
SELECT 
        FlightDate,
        Reporting_Airline,
        Flight_Number_Reporting_Airline,
        Origin,
        OriginCityName,
        Dest,
        DestCityName,
        CRSDepTime,
        DepTime,
        DepDelayMinutes,
        CRSArrTime,
        ArrTime,
        ArrDelayMinutes,
        Cancelled,
        CancellationCode,
        Diverted,
        DivAirportLandings,
        ActualElapsedTime,
        AirTime,
        CarrierDelay,
        WeatherDelay,
        NASDelay,
        SecurityDelay,
        LateAircraftDelay
    FROM hive_metastore.default.ontime_cleaned
    WHERE Reporting_Airline = 'OO'
      AND Flight_Number_Reporting_Airline = 4277
      AND FlightDate = '2023-12-30'
      AND Cancelled=1
    ORDER BY Origin, Dest

FlightDate,Reporting_Airline,Flight_Number_Reporting_Airline,Origin,OriginCityName,Dest,DestCityName,CRSDepTime,DepTime,DepDelayMinutes,CRSArrTime,ArrTime,ArrDelayMinutes,Cancelled,CancellationCode,Diverted,DivAirportLandings,ActualElapsedTime,AirTime,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
2023-12-30,OO,4277,CDC,"Cedar City, UT",SLC,"Salt Lake City, UT",1249,null,null,1356,null,null,1.0,B,0.0,0,null,null,null,null,null,null,null
2023-12-30,OO,4277,SLC,"Salt Lake City, UT",CDC,"Cedar City, UT",1100,null,null,1209,null,null,1.0,B,0.0,0,null,null,null,null,null,null,null


## Future Work

- Integration with Evaluation Framwork (DeepEvals)
- SparkSession was hanging on query, need to debug that.
- Test Context Memory